# Проміжний урок: розрізи складальних одиниць

Інтерактивна демонстраційна модель монтажного вузла: основа, L-подібний кронштейн, два болти, шайби й гайки.

**Мета:** побачити зв'язок між просторовою складальною одиницею та її зображенням у розрізі; навчитися аналізувати положення січної площини.

Це навчальна геометрична візуалізація, а не повний генератор креслень за стандартами. Штрихування та умовності спрощені.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Polygon
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import ipywidgets as widgets
from IPython.display import display

L,W,Tb,T,H = 160,90,10,8,75
x0=22
bolt_x=[65,125]
bolt_r,washer_r,washer_t,nut_h=4,8,2,6

def box(ax,x1,x2,y1,y2,z1,z2,color,alpha=.95):
    v=[(x1,y1,z1),(x2,y1,z1),(x2,y2,z1),(x1,y2,z1),
       (x1,y1,z2),(x2,y1,z2),(x2,y2,z2),(x1,y2,z2)]
    f=[[v[0],v[1],v[2],v[3]],[v[4],v[5],v[6],v[7]],
       [v[0],v[1],v[5],v[4]],[v[1],v[2],v[6],v[5]],
       [v[2],v[3],v[7],v[6]],[v[3],v[0],v[4],v[7]]]
    ax.add_collection3d(Poly3DCollection(f,facecolors=color,edgecolors="#263746",linewidths=.7,alpha=alpha))

def cylinder_y(ax,x,z,y1,y2,r,color):
    a=np.linspace(0,2*np.pi,40)
    A,Y=np.meshgrid(a,[y1,y2])
    ax.plot_surface(x+r*np.cos(A),Y,z+r*np.sin(A),color=color,linewidth=0,alpha=.98)
    for y in [y1,y2]:
        ax.plot(x+r*np.cos(a),np.full_like(a,y),z+r*np.sin(a),color="#263746",lw=.5)

def hex_nut(ax,x,z,y1,y2,r):
    a=np.linspace(0,2*np.pi,7)[:-1]+np.pi/6
    p=[(x+r*np.cos(t),z+r*np.sin(t)) for t in a]
    lo=[(xx,y1,zz) for xx,zz in p]; hi=[(xx,y2,zz) for xx,zz in p]
    faces=[lo,hi]+[[lo[i],lo[(i+1)%6],hi[(i+1)%6],hi[i]] for i in range(6)]
    ax.add_collection3d(Poly3DCollection(faces,facecolors="#b5bbc1",edgecolors="#263746",linewidths=.6))

def draw_assembly(mode="Поздовжня (XY)",position=45):
    fig=plt.figure(figsize=(15,6.5))
    a3=fig.add_subplot(121,projection="3d"); a2=fig.add_subplot(122)
    # assembly components
    box(a3,0,L,0,Tb,0,W,"#a8b8c6")
    box(a3,x0,L-12,Tb,Tb+T,0,W,"#d5a45b")
    box(a3,x0,x0+T,Tb+T,Tb+T+H,0,W,"#d5a45b")
    for x in bolt_x:
        cylinder_y(a3,x,W/2,-nut_h,Tb+T+washer_t+2,bolt_r,"#687987")
        cylinder_y(a3,x,W/2,Tb+T,Tb+T+washer_t,washer_r,"#c3c8ce")
        hex_nut(a3,x,W/2,-nut_h,0,bolt_r*1.7)
    # cutting plane
    if mode=="Поздовжня (XY)":
        xx,yy=np.meshgrid([0,L],[0,Tb+T+H]); zz=np.full_like(xx,position)
        a3.plot_surface(xx,yy,zz,color="#efb84c",alpha=.22,shade=False)
    else:
        xx,zz=np.meshgrid([0,L],[0,W]); yy=np.full_like(xx,position)
        a3.plot_surface(xx,yy,zz,color="#efb84c",alpha=.22,shade=False)
    a3.set(xlim=(0,L),ylim=(-nut_h-3,Tb+T+H+5),zlim=(0,W),
           xlabel="X — довжина",ylabel="Y — висота",zlabel="Z — ширина",
           title="Складальна одиниця та січна площина")
    a3.set_box_aspect((L,Tb+T+H+nut_h,W)); a3.view_init(elev=22,azim=-55)

    if mode=="Поздовжня (XY)":
        z=position
        a2.add_patch(Rectangle((0,0),L,Tb,facecolor="#dce3e8",edgecolor="black",hatch="////"))
        prof=[(x0,Tb),(L-12,Tb),(L-12,Tb+T),(x0+T,Tb+T),(x0+T,Tb+T+H),(x0,Tb+T+H)]
        a2.add_patch(Polygon(prof,closed=True,facecolor="#f2d9ad",edgecolor="black",hatch="\\\\"))
        if abs(z-W/2)<.8:
            for x in bolt_x:
                a2.add_patch(Rectangle((x-bolt_r,Tb-nut_h),2*bolt_r,T+Tb+washer_t+nut_h,
                                       facecolor="#c5cbd0",edgecolor="black"))
                a2.add_patch(Rectangle((x-washer_r,Tb+T),2*washer_r,washer_t,
                                       facecolor="white",edgecolor="black"))
                a2.add_patch(Rectangle((x-bolt_r*1.7,-nut_h),bolt_r*3.4,nut_h,
                                       facecolor="#e2e5e8",edgecolor="black"))
            note="Площина проходить через осі болтів"
        else:
            for x in bolt_x: a2.plot(x,Tb+T+3,"x",color="darkred",ms=8)
            note="Хрестики — осі болтів поза площиною розрізу"
        a2.set(xlim=(-8,L+8),ylim=(-nut_h-5,Tb+T+H+10),
               xlabel="X — довжина",ylabel="Y — висота",
               title=f"Поздовжній розріз XY, Z={z:.1f} мм\n{note}")
    else:
        y=position
        if 0<=y<=Tb:
            a2.add_patch(Rectangle((0,0),L,W,facecolor="#dce3e8",edgecolor="black",hatch="////"))
            desc="Перетинається основа"
        elif Tb<y<=Tb+T:
            a2.add_patch(Rectangle((x0,0),L-12-x0,W,facecolor="#f2d9ad",edgecolor="black",hatch="\\\\"))
            for x in bolt_x: a2.add_patch(Circle((x,W/2),bolt_r,facecolor="white",edgecolor="black"))
            desc="Перетинається горизонтальна полиця"
        elif Tb+T<y<=Tb+T+H:
            a2.add_patch(Rectangle((x0,0),T,W,facecolor="#f2d9ad",edgecolor="black",hatch="\\\\"))
            desc="Перетинається вертикальна полиця"
        else: desc="Площина вище вузла"
        a2.set(xlim=(-8,L+8),ylim=(-8,W+8),aspect="equal",
               xlabel="X — довжина",ylabel="Z — ширина",
               title=f"Горизонтальний розріз XZ, Y={y:.1f} мм\n{desc}")
    a2.grid(True,alpha=.25); plt.tight_layout(); plt.show()

draw_assembly()


## 1. Інтерактивне переміщення січної площини

- Поздовжня площина XY: змінюйте координату `Z`. При `Z = W/2` площина проходить через осі двох болтів.
- Горизонтальна площина XZ: змінюйте координату `Y` від основи до верхньої частини кронштейна.

In [ ]:
mode=widgets.ToggleButtons(options=["Поздовжня (XY)","Горизонтальна (XZ)"],description="Площина:")
pos=widgets.FloatSlider(value=W/2,min=0,max=W,step=1,description="Z, мм")
def change_mode(change):
    if mode.value=="Поздовжня (XY)":
        pos.min,pos.max,pos.value,pos.description=0,W,W/2,"Z, мм"
    else:
        pos.min,pos.max,pos.value,pos.description=-nut_h,Tb+T+H+5,Tb+T/2,"Y, мм"
mode.observe(change_mode,names="value")
out=widgets.interactive_output(draw_assembly,{"mode":mode,"position":pos})
display(widgets.VBox([mode,pos]),out)


## 2. Що спостерігати

1. Січна площина умовно розтинає об'єкт; частину між спостерігачем і площиною подумки відкидають.
2. На розрізі показують фігуру в січній площині та видимі елементи за нею.
3. Перетятий матеріал деталей позначають штрихуванням.
4. Суміжні деталі розрізняють напрямком або кроком штрихування.
5. Стандартні кріпильні деталі (болти, гайки, шайби, шпильки тощо), якщо площина проходить уздовж їхньої осі, за загальним правилом показують нерозсіченими.
6. Розріз і переріз не тотожні: розріз показує фігуру в січній площині та видиме за нею; переріз — лише фігуру, що утворилася в площині.

**Примітка:** для нормативного оформлення креслень необхідно звіряти правила та винятки з чинними стандартами, що застосовуються у курсі.

## 3. Практичні завдання

1. Встановіть поздовжню площину на `Z=W/2`. Назвіть компоненти, які потрапили в розріз.
2. Перемістіть площину до краю. Чому болти перестають відображатися в розрізі?
3. Переміщуйте горизонтальну площину крізь основу, горизонтальну та вертикальну полиці. Опишіть зміну фігури перетину.
4. Поясніть, чому кріпильні деталі в поздовжньому розрізі не штрихуються, коли площина проходить уздовж їхніх осей.
5. Намалюйте ескіз одного з розрізів і позначте положення січної площини.

## 4. Контрольні запитання

1. Для чого застосовують розрізи складальних одиниць?
2. Чим розріз відрізняється від перерізу?
3. Що таке січна площина?
4. Які деталі штрихують у складальному розрізі?
5. Які стандартні кріпильні деталі зазвичай показують нерозсіченими, якщо площина проходить уздовж їхньої осі?
6. Як змінюється зображення при переміщенні січної площини?

**Очікуваний результат:** студент розуміє зв'язок між просторовою складальною одиницею та її розрізом, може пояснити вибір січної площини та основні умовності зображення.